In [1]:
import os
import gc
import random
import math
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings('ignore')

# Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if device.type == "cuda":
    torch.cuda.manual_seed_all(SEED)

# ==========================================
# 1. DATA LOADING
# ==========================================
DATA_DIR = "/kaggle/input/competitions/m5-forecasting-accuracy"

print("Loading M5 datasets...")
sales = pd.read_csv(os.path.join(DATA_DIR, "sales_train_evaluation.csv"))
sample_sub = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))

# Store validation IDs for RMSE mapping later
validation_ids = sales['id'].str.replace('_evaluation', '_validation').values

# ==========================================
# 2. MATRIX BUILDING & SCALING
# ==========================================
# Keep only day columns: d_1 ... d_1941
day_cols = [c for c in sales.columns if c.startswith("d_")]
day_cols = sorted(day_cols, key=lambda x: int(x.split("_")[1]))

values = sales[day_cols].values.astype("float32")
n_series, n_days = values.shape

# Official M5 Split Setup
LAST_TRAIN_DAY = 1885   # Stop at 1885 so we can validate on 1886-1913
HORIZON = 28            # forecast horizon (28 days)
HISTORY = 90            # input history length for the model

# Per-series scaling (Avoid exploding gradients)
train_region = values[:, :LAST_TRAIN_DAY]          
series_means = train_region.mean(axis=1)           

series_scales = series_means.copy()
series_scales[series_scales < 1.0] = 1.0           
values_scaled = values / series_scales[:, None]    

# ==========================================
# 3. DATASET & DATALOADER
# ==========================================
class M5NBeatsDataset(Dataset):
    def __init__(self, all_series, history, horizon, last_train_day, samples_per_series=20):
        self.all_series = all_series
        self.history = history
        self.horizon = horizon
        self.last_train_day = last_train_day
        self.samples = []

        n_series, n_days = all_series.shape
        max_start = last_train_day - history - horizon + 1
        max_start = max(max_start, 0)

        for s in range(n_series):
            for _ in range(samples_per_series):
                start = 0 if max_start <= 0 else np.random.randint(0, max_start + 1)
                self.samples.append((s, start))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s, start = self.samples[idx]
        x = self.all_series[s, start : start + self.history]
        y = self.all_series[s, start + self.history : start + self.history + self.horizon]
        return torch.from_numpy(x), torch.from_numpy(y)

train_dataset = M5NBeatsDataset(
    all_series=values_scaled,
    history=HISTORY,
    horizon=HORIZON,
    last_train_day=LAST_TRAIN_DAY,
    samples_per_series=10,
)

BATCH_SIZE = 1024
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,
    num_workers=2,
)

# ==========================================
# 4. N-BEATS ARCHITECTURE
# ==========================================
class NBeatsBlock(nn.Module):
    def __init__(self, input_size, theta_size, hidden_size=256, nb_hid_layers=4, dropout_p=0.1):
        super().__init__()
        layers = []
        in_features = input_size
        for _ in range(nb_hid_layers):
            layers.append(nn.Linear(in_features, hidden_size))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(p=dropout_p))
            in_features = hidden_size
        self.fc = nn.Sequential(*layers)
        self.backcast_linear = nn.Linear(hidden_size, theta_size)
        self.forecast_linear = nn.Linear(hidden_size, theta_size)

    def forward(self, x):
        x = self.fc(x)
        return self.backcast_linear(x), self.forecast_linear(x)

class NBeats(nn.Module):
    def __init__(self, history, horizon, n_stacks=3, hidden_size=256, nb_hid_layers=4, dropout_p=0.1):
        super().__init__() # FIXED INDENTATION
        self.history = history
        self.horizon = horizon
        self.stacks = nn.ModuleList([
            NBeatsBlock(history, history, hidden_size, nb_hid_layers, dropout_p)
            for _ in range(n_stacks)
        ])
        self.forecast_head = nn.Linear(history, horizon)

    def forward(self, x):
        residual = x
        for block in self.stacks:
            backcast, _ = block(residual)
            residual = residual - backcast
        return self.forecast_head(residual)

model = NBeats(history=HISTORY, horizon=HORIZON).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)

# ==========================================
# 5. TRAINING LOOP
# ==========================================
EPOCHS = 60

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    for batch_x, batch_y in loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer.zero_grad()
        output = model(batch_x)
        loss = criterion(output, batch_y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item() * batch_x.size(0)
    return total_loss / len(loader.dataset)

print("\n--- Commencing N-BEATS Training ---")
for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion)
    print(f"Epoch {epoch}/{EPOCHS} - train_loss: {train_loss:.6f}")

# ==========================================
# 6. VALIDATION INFERENCE (Days 1886 - 1913)
# ==========================================
print("\n--- Calculating Validation Item-Level Metrics ---")
model.eval()
val_forecasts = np.zeros((n_series, HORIZON), dtype="float32")
actual_val_sales = values[:, LAST_TRAIN_DAY : LAST_TRAIN_DAY + HORIZON]

with torch.no_grad():
    for i in range(n_series):
        start = LAST_TRAIN_DAY - HISTORY
        x = values_scaled[i, start:LAST_TRAIN_DAY]
        x_tensor = torch.from_numpy(x).unsqueeze(0).to(device)
        forecast_scaled = model(x_tensor).cpu().numpy().reshape(-1)
        
        forecast = forecast_scaled * series_scales[i]
        val_forecasts[i, :] = np.clip(forecast, 0.0, None)

# Print Global Validation RMSE
global_rmse = np.sqrt(mean_squared_error(actual_val_sales.flatten(), val_forecasts.flatten()))
print(f"Our final global N-BEATS val rmse score is {global_rmse:.4f}")

# ==========================================
# 7. KAGGLE EVALUATION INFERENCE (Days 1914 - 1941)
# ==========================================
print("\n--- Generating Kaggle Submission ---")
eval_forecasts = np.zeros((n_series, HORIZON), dtype="float32")
TRUE_EVAL_START = 1913 # Days 1914-1941

with torch.no_grad():
    for i in range(n_series):
        start = TRUE_EVAL_START - HISTORY
        x = values_scaled[i, start:TRUE_EVAL_START]
        x_tensor = torch.from_numpy(x).unsqueeze(0).to(device)
        forecast_scaled = model(x_tensor).cpu().numpy().reshape(-1)
        
        forecast = forecast_scaled * series_scales[i]
        eval_forecasts[i, :] = np.clip(forecast, 0.0, None)

# Fill submission dataframe
sub = sample_sub.copy()
val_mask = sub["id"].str.endswith("_validation")
eval_mask = sub["id"].str.endswith("_evaluation")
f_cols = [f"F{i}" for i in range(1, HORIZON + 1)]

# Put the 1886-1913 predictions in the validation rows
sub.loc[val_mask, f_cols] = val_forecasts
# Put the 1914-1941 predictions in the evaluation rows
sub.loc[eval_mask, f_cols] = eval_forecasts

sub.to_csv("submission.csv", index=False)
print("Saved final submission.csv successfully!")

# ==========================================
# 6. VALIDATION INFERENCE (Days 1886 - 1913)
# ==========================================
print("\n--- Calculating Validation Item-Level Metrics ---")
model.eval()
val_forecasts = np.zeros((n_series, HORIZON), dtype="float32")
actual_val_sales = values[:, LAST_TRAIN_DAY : LAST_TRAIN_DAY + HORIZON]

with torch.no_grad():
    for i in range(n_series):
        start = LAST_TRAIN_DAY - HISTORY
        x = values_scaled[i, start:LAST_TRAIN_DAY]
        x_tensor = torch.from_numpy(x).unsqueeze(0).to(device)
        forecast_scaled = model(x_tensor).cpu().numpy().reshape(-1)
        
        forecast = forecast_scaled * series_scales[i]
        val_forecasts[i, :] = np.clip(forecast, 0.0, None)

# --- NEW: Calculate and Save Item-Level RMSE ---
item_results = []
for i in range(n_series):
    mse = mean_squared_error(actual_val_sales[i], val_forecasts[i])
    item_results.append({'id': validation_ids[i], 'nbeats_rmse': np.sqrt(mse)})

item_rmse_df = pd.DataFrame(item_results)
item_rmse_df.to_csv('nbeats_item_level_rmse.csv', index=False)
print(f"Successfully saved item-level RMSE for {len(item_rmse_df)} products to 'nbeats_item_level_rmse.csv'")

# Print Global Validation RMSE
global_rmse = np.sqrt(mean_squared_error(actual_val_sales.flatten(), val_forecasts.flatten()))
print(f"Our final global N-BEATS val rmse score is {global_rmse:.4f}")

Using device: cuda
Loading M5 datasets...

--- Commencing N-BEATS Training ---
Epoch 1/60 - train_loss: 0.895980
Epoch 2/60 - train_loss: 0.831102
Epoch 3/60 - train_loss: 0.825894
Epoch 4/60 - train_loss: 0.822579
Epoch 5/60 - train_loss: 0.820206
Epoch 6/60 - train_loss: 0.819319
Epoch 7/60 - train_loss: 0.818774
Epoch 8/60 - train_loss: 0.817120
Epoch 9/60 - train_loss: 0.815412
Epoch 10/60 - train_loss: 0.813663
Epoch 11/60 - train_loss: 0.812819
Epoch 12/60 - train_loss: 0.811248
Epoch 13/60 - train_loss: 0.809401
Epoch 14/60 - train_loss: 0.808432
Epoch 15/60 - train_loss: 0.806922
Epoch 16/60 - train_loss: 0.805433
Epoch 17/60 - train_loss: 0.803299
Epoch 18/60 - train_loss: 0.801652
Epoch 19/60 - train_loss: 0.800818
Epoch 20/60 - train_loss: 0.799283
Epoch 21/60 - train_loss: 0.798147
Epoch 22/60 - train_loss: 0.794696
Epoch 23/60 - train_loss: 0.794278
Epoch 24/60 - train_loss: 0.793269
Epoch 25/60 - train_loss: 0.790740
Epoch 26/60 - train_loss: 0.789867
Epoch 27/60 - train_